# AMAN v1.4 — controlled synthetic candidate

Use a fresh **hosted Colab CPU runtime**. Upload the unchanged `AMAN_Colab_Training_v1.4_FINAL.zip` in the Files sidebar and wait until upload finishes. Then run cells from top to bottom.

No GPU or Colab Pro is required. Training is on Google's server, not your laptop (your browser still uses local resources). The dataset is synthetic, not real-world safety validation. No candidate is automatically promoted.

The notebook keeps Colab NumPy/pandas untouched. It checks the ZIP and every member, runs nine bounded classifier ablations, selects calibration and alert policy on separate runs, freezes everything, then compares against actual saved v1.2 and v1.3.

Default local Colab storage is temporary. Download both ZIPs promptly. Optional Drive output preserves a frozen checkpoint across runtime loss; without it, resume only works while `/content` survives. Before freeze, an interrupted fit restarts from the beginning; after freeze, matching hashes/config allow evaluation-only resume.

In [ ]:
#@title 1. Configuration — keep these defaults for a sidebar upload
BUNDLE_SOURCE = 'sidebar' #@param ['sidebar', 'drive']
DRIVE_BUNDLE = '/content/drive/MyDrive/AMAN/AMAN_Colab_Training_v1.4_FINAL.zip' #@param {type:'string'}
SAVE_OUTPUT_TO_DRIVE = False #@param {type:'boolean'}
DRIVE_OUTPUT = '/content/drive/MyDrive/AMAN/training_runs/run_v1_4' #@param {type:'string'}
BOOTSTRAP_REPLICATES = 200 #@param {type:'integer'}
SEED = 20260918
COMPUTE = 'cpu'
EXPECTED_BYTES = 243655088
EXPECTED_SHA256 = 'f3b34e55c3a97f08da361210f18ec05a90ca27cbcf75917913780e5977c8fbf2'
assert BOOTSTRAP_REPLICATES >= 50
print('CPU training; no GPU needed. Wait for the ZIP upload to finish.')

In [ ]:
#@title 2. Locate and completely verify the training ZIP
from pathlib import Path
import hashlib, time

if BUNDLE_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    archive = Path(DRIVE_BUNDLE)
else:
    archive = Path('/content/AMAN_Colab_Training_v1.4_FINAL.zip')
    if not archive.is_file():
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly AMAN_Colab_Training_v1.4_FINAL.zip')
        archive = Path('/content') / next(iter(uploaded))
        del uploaded
if not archive.is_file():
    raise FileNotFoundError(f'Training ZIP not found: {archive}')

# A sidebar upload can appear before all bytes are flushed. Require stable exact size.
sizes = []
for _ in range(3):
    sizes.append(archive.stat().st_size)
    time.sleep(2)
if len(set(sizes)) != 1 or sizes[0] != EXPECTED_BYTES:
    raise RuntimeError(f'ZIP incomplete/wrong: sizes {sizes}; expected {EXPECTED_BYTES}. '
                       'Wait for upload to finish, or delete and upload the ZIP again.')
h = hashlib.sha256()
with archive.open('rb') as handle:
    for block in iter(lambda: handle.read(1 << 20), b''):
        h.update(block)
archive_hash = h.hexdigest()
if archive_hash != EXPECTED_SHA256:
    raise RuntimeError(f'Wrong/corrupted ZIP: {archive_hash}; expected {EXPECTED_SHA256}')
print(f'ZIP verified: {archive.name}, {sizes[0]:,} bytes, SHA-256 {archive_hash}')

In [ ]:
#@title 3. Safely extract and verify every bundled file
import json, shutil, tempfile, zipfile
from pathlib import PurePosixPath

extract_root = Path(tempfile.mkdtemp(prefix='aman-v14-bundle-', dir='/content'))
try:
    with zipfile.ZipFile(archive) as z:
        infos = z.infolist()
        names = [item.filename for item in infos]
        if len(names) != len(set(names)):
            raise ValueError('Duplicate ZIP entries')
        if sum(item.file_size for item in infos) > 2_000_000_000:
            raise ValueError('Unexpected oversized archive')
        for item in infos:
            relative = PurePosixPath(item.filename)
            if chr(92) in item.filename or ':' in item.filename or relative.is_absolute() or '..' in relative.parts:
                raise ValueError(f'Unsafe archive path: {item.filename}')
            target = extract_root.joinpath(*relative.parts)
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            try:
                with z.open(item) as source, target.open('wb') as destination:
                    shutil.copyfileobj(source, destination, length=1 << 20)
            except Exception as error:
                raise RuntimeError(f'Could not extract {item.filename}; ZIP may still be uploading') from error
except Exception:
    shutil.rmtree(extract_root, ignore_errors=True)
    raise

manifest = json.loads((extract_root/'bundle_manifest.json').read_text(encoding='utf-8'))
actual = {str(p.relative_to(extract_root)).replace(chr(92),'/') for p in extract_root.rglob('*') if p.is_file()}
expected = set(manifest['files']) | {'bundle_manifest.json'}
if actual != expected:
    raise RuntimeError(f'Inventory mismatch; missing={sorted(expected-actual)}, extra={sorted(actual-expected)}')
for name, expected_hash in manifest['files'].items():
    target = (extract_root/name).resolve()
    if not target.is_relative_to(extract_root):
        raise ValueError(f'Unsafe manifest path: {name}')
    digest = hashlib.sha256()
    with target.open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    if digest.hexdigest() != expected_hash:
        raise RuntimeError(f'Bundled-file hash mismatch: {name}')
print(f'Extraction complete: all {len(manifest["files"])} files verified in {extract_root}')

In [ ]:
#@title 4. Check the stock Colab stack; isolate only missing LightGBM
import os, subprocess, sys

def run_visible(command, label, env=None):
    print(label, flush=True)
    process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'{label} failed (exit {code}); full error is printed above.')

training_python = Path(sys.executable)
dependency_dir = Path(tempfile.mkdtemp(prefix='aman-v14-deps-', dir='/content'))
dependency_env = os.environ.copy()
dependency_env['PYTHONPATH'] = str(dependency_dir)
# Abort visibly on a broken base image; do not guess NumPy/pandas replacements.
run_visible([str(training_python), '-c',
    'import numpy,pandas,pyarrow,scipy,sklearn,matplotlib; '
    'print("Stock stack:",numpy.__version__,pandas.__version__,pyarrow.__version__)'],
    'Verify untouched Colab scientific stack', dependency_env)
probe = subprocess.run([str(training_python), '-c',
    'import lightgbm; assert lightgbm.__version__ == "4.6.0"'], env=dependency_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if probe.returncode:
    print('LightGBM 4.6.0 not usable in stock environment; installing isolated wheel.')
    run_visible([str(training_python), '-m', 'pip', 'install', '--target', str(dependency_dir),
        '--no-deps', '--only-binary=:all:', '--disable-pip-version-check', '--retries', '10',
        '--timeout', '120', 'lightgbm==4.6.0'], 'Install isolated LightGBM', dependency_env)
run_visible([str(training_python), '-c',
    'import lightgbm,numpy,pandas,pyarrow,scipy,sklearn,google.colab; '
    'print("Verified:",lightgbm.__version__,numpy.__version__,pandas.__version__)'],
    'Verify complete training runtime', dependency_env)
print('Ready. Colab NumPy/pandas were not changed.')

In [ ]:
#@title 5. Train, freeze, evaluate and export (safe to rerun)
if SAVE_OUTPUT_TO_DRIVE:
    if not Path('/content/drive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    output = Path(DRIVE_OUTPUT)
else:
    output = Path('/content/AMAN_v14_training_output')

configuration = Path('/content/AMAN_v14_training_config.json')
configuration.write_text(json.dumps(dict(seed=SEED, compute=COMPUTE,
    bootstrap_replicates=BOOTSTRAP_REPLICATES)), encoding='utf-8')
process_env = os.environ.copy()
process_env['PYTHONPATH'] = str(dependency_dir) + os.pathsep + str(extract_root/'runtime')
process_env['MPLBACKEND'] = 'Agg'
command = [str(training_python), '-u', '-m', 'aman_ml.run_v14',
           '--bundle', str(extract_root), '--output', str(output), '--config', str(configuration)]
run_visible(command, 'AMAN v1.4 training and paired evaluation', process_env)
print(f'Completed successfully. Outputs: {output}')

In [ ]:
#@title 6. Review the frozen comparisons — no automatic promotion
metrics_path = output/'metrics.json'
if not (output/'AMAN_Training_Report_v1.4.zip').is_file():
    raise RuntimeError('Evaluation/export incomplete. Do not interpret partial metrics as final.')
metrics = json.loads(metrics_path.read_text())
for split, versions in metrics.items():
    print('\n' + split.upper())
    for name, result in versions.items():
        print(name, 'AP=', round(result['row']['pr_auc'],4),
              'event recall=', round(result['operational']['critical_event_recall'],4),
              'false episodes/scored zone-hour=', round(result['operational']['false_early_warnings_per_zone_hour'],4),
              'p90 coverage=', round(result['density']['p90_coverage'],4))
print(json.dumps(json.loads((output/'promotion_review.json').read_text()), indent=2))
print('Send both output ZIPs for independent review before replacing v1.2.')

In [ ]:
#@title 7. Recover and download both result ZIPs (safe after lost variables)
from pathlib import Path
import hashlib, zipfile
from google.colab import files

names = ['AMAN_Inference_Artifacts_v1.4.zip', 'AMAN_Training_Report_v1.4.zip']
roots = [Path('/content')]
if Path('/content/drive').is_dir():
    roots.append(Path('/content/drive/MyDrive'))
found = {}
for name in names:
    matches = []
    for root in roots:
        matches.extend(p for p in root.rglob(name) if p.is_file())
    if not matches:
        raise FileNotFoundError(
            f'{name} was not found under /content or mounted Drive. '
            'If the runtime restarted and you did not save to Drive, rerun cell 5.')
    result = max(matches, key=lambda p: p.stat().st_mtime)
    with zipfile.ZipFile(result) as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(f'Corrupt ZIP member in {result}: {bad}')
    digest = hashlib.sha256(result.read_bytes()).hexdigest()
    found[name] = result
    print(f'Verified: {result} ({result.stat().st_size/1e6:.1f} MB), SHA-256 {digest}')

# One browser download is more reliable than triggering two simultaneously.
review = Path('/content/AMAN_v1.4_RESULTS_FOR_REVIEW.zip')
with zipfile.ZipFile(review, 'w', zipfile.ZIP_STORED) as z:
    for name, result in found.items():
        z.write(result, name)
print(f'Downloading combined review package: {review} ({review.stat().st_size/1e6:.1f} MB)')
files.download(str(review))